# 03 - Retrieval

**Goal:** given a question, find the chunks most likely to contain the answer.

**How it works**
1. Embed the question with the same model
2. Compare it with every chunk embedding (dot product)
3. Take the chunks with the highest scores

This is the "R" in RAG. If retrieval is bad, the LLM will give bad answers, so we test it carefully here
before adding any LLM.

**Steps**
1. Load chunks, embeddings and the model
2. Write the `retrieve` function
3. Try it on real questions
4. See what happens with questions that are NOT in the papers
5. Tune `top_k`
6. A first mini evaluation

## Step 1 - Load everything we saved

In [ ]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

PROCESSED_DIR = Path("../data/processed")

# The text pieces (from notebook 01)
with open(PROCESSED_DIR / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

# The vectors (from notebook 02)
embeddings = np.load(PROCESSED_DIR / "embeddings.npy")

# The SAME model we used for the chunks (very important!)
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"{len(chunks)} chunks, embeddings shape {embeddings.shape}")
assert len(chunks) == len(embeddings), "chunks and embeddings must match!"

## Step 2 - The retrieve function

In [ ]:
def retrieve(question, top_k=4):
    # 1. Turn the question into a vector
    q_vec = model.encode([question], normalize_embeddings=True)[0]

    # 2. Similarity with EVERY chunk at once (one dot product per chunk)
    scores = embeddings @ q_vec

    # 3. Indices of the best scores, highest first
    best_indices = np.argsort(scores)[::-1][:top_k]

    # 4. Return the chunk text + where it came from + its score
    results = []
    for i in best_indices:
        results.append({
            "text": chunks[i]["text"],
            "source": chunks[i]["source"],
            "page": chunks[i]["page"],
            "score": float(scores[i]),
        })
    return results

# NOTE: this is exactly the format the Streamlit app expects from retrieve_chunks()

## Step 3 - Try it on real questions

In [ ]:
def show(results):
    # Print results in a readable way
    for rank, r in enumerate(results, start=1):
        print(f"#{rank}  score={r['score']:.2f}  {r['source']}  (page {r['page']})")
        print("    " + r["text"][:300] + "...")
        print()

# CHANGE THIS to a question that your papers can answer
question = "What is the main contribution of this paper?"

show(retrieve(question, top_k=4))

# TEACHING POINT: read the top results. Do they actually answer the question?
# This manual check is the most valuable skill in building RAG systems.

## Step 4 - What if the answer is NOT in the papers?

Retrieval ALWAYS returns something, even for nonsense questions. Look at the scores.

In [ ]:
off_topic = "What is the best recipe for chapati?"

show(retrieve(off_topic, top_k=3))

# Compare these scores with Step 3. Low scores = the papers probably do not contain the answer.
# Many systems add a minimum score so weak matches are thrown away.

In [ ]:
def retrieve_with_threshold(question, top_k=4, min_score=0.25):
    # Same as retrieve(), but drops weak matches
    results = retrieve(question, top_k)
    return [r for r in results if r["score"] >= min_score]

print("On-topic results :", len(retrieve_with_threshold(question)))
print("Off-topic results:", len(retrieve_with_threshold(off_topic)))

# EXERCISE: find a good min_score for YOUR papers by testing several questions.

## Step 5 - Tune `top_k`

In [ ]:
for k in [1, 3, 5, 10]:
    results = retrieve(question, top_k=k)
    print(f"top_k={k:>2}  ->  best score {results[0]['score']:.2f}, "
          f"worst score {results[-1]['score']:.2f}")

# DISCUSS:
# - Small top_k : short prompt, but we may miss the right chunk
# - Large top_k : more chances to include it, but more noise and a longer prompt

## Step 6 - A first mini evaluation

How do we know retrieval is *good* and not just *looks okay*? We test it.

Idea: write questions where you **know a keyword that must appear** in the right chunk.
Then check how often the right chunk is found in the top results (this is called the **hit rate**).

In [ ]:
# EDIT THIS LIST using your own papers: (question, a keyword the correct chunk must contain)
test_set = [
    ("What dataset was used?", "dataset"),
    ("What evaluation metric was used?", "accuracy"),
    ("What are the limitations?", "limitation"),
]

def hit_rate(test_set, top_k=4):
    hits = 0
    for q, keyword in test_set:
        results = retrieve(q, top_k)
        found = any(keyword.lower() in r["text"].lower() for r in results)
        hits += found
        print(("HIT  " if found else "MISS ") + q)
    return hits / len(test_set)

rate = hit_rate(test_set, top_k=4)
print(f"\nHit rate: {rate:.0%}")

# EXERCISE: run it again with top_k=1, 3, 8. How does the hit rate change?
# This is the seed of the src/evaluation folder.

## Exercises for students
1. Write 5 real test questions for your papers and measure the hit rate.
2. Change the chunk size in notebook 01, re-run 02 and 03. Did the hit rate improve?
3. Why must the question and the chunks use the *same* embedding model?
4. Print the 10 results for one question. At which rank does the answer appear?

**Next steps**
- Generation: pass the retrieved chunks and the question to an LLM.
- Move `clean_text`, `chunk_text`, the embedding code and `retrieve` into `src/`.
- Connect them to the Streamlit placeholders.